# معمل تأسيس البيئة الافتراضية وحوكمة المفاتيح باستخدام UV

أهلاً بك في المعمل العملي للمحاضرة 58!

في المحاضرة دي اتعلمنا الخطوات العملية اللي بنعملها في بداية أي مشروع ذكاء اصطناعي جديد عشان بيئتنا تكون نظيفة ومشتركة وآمنة:
1. فحص ملف `pyproject.toml` ومراقبة المكتبات اللي أضافتها أداة `uv`.
2. التأكد من وجود نواة `ipykernel` لتشغيل نوت بوك Jupyter بسلاسة.
3. قراءة مفاتيح الـ API بأمان من ملف `.env` باستخدام `python-dotenv` وإخفاء أجزائها الحساسة عند الطباعة.


In [1]:
# تجهيز المكتبات وفحص نواة Jupyter
import os
import tomllib
from pathlib import Path
import ipykernel
from dotenv import load_dotenv

print("نواة ipykernel متاحة بإصدار:", ipykernel.__version__)


نواة ipykernel متاحة بإصدار: 7.4.0


### 1. قراءة ملف `pyproject.toml` بتاع المشروع

لاحظ هنا إزاي أداة `uv` لما بنكتب `uv add -r requirements.txt` بتسجل كل مكتبة بإصدارها جوا ملف `pyproject.toml`. تعال نفتح الملف ونطبع مكتبات LangChain الموجودة فيه!


In [2]:
# البحث عن ملف pyproject.toml في المشروع وقراءته
toml_path = next((p / "pyproject.toml" for p in [Path("."), Path(".."), Path("../..")] if (p / "pyproject.toml").exists()), None)

if toml_path:
    with open(toml_path, "rb") as f:
        config = tomllib.load(f)
    deps = config["project"]["dependencies"]
    print(f"اسم المشروع: {config['project']['name']} | إجمالي المكتبات: {len(deps)}")
    print("مكتبات LangChain المسجلة:")
    for d in deps:
        if "langchain" in d or "dotenv" in d:
            print("  -", d)


اسم المشروع: rag-mastery | إجمالي المكتبات: 24
مكتبات LangChain المسجلة:
  - langchain>=0.3.0
  - langchain-community>=0.3.0
  - langchain-core>=0.3.0
  - langchain-experimental>=0.4.2
  - langchain-groq>=1.1.3
  - langchain-huggingface>=0.1.2
  - langchain-openai>=0.2.0
  - langchain-text-splitters>=0.3.0
  - python-dotenv>=1.0.1


### 2. فحص مفاتيح الـ API بأمان (`python-dotenv`)

في الفيديو ضفنا 3 مفاتيح: Google Gemini و Groq و OpenAI في ملف `.env`.
تعال نحمل ملف `.env` ونتأكد من المفاتيح الموجودة حالياً في جهازك، مع عمل تغطية (Masking) عشان ما نظهرش المفتاح الحقيقي على الشاشة!


In [3]:
load_dotenv()

def mask_key(value):
    if not value or len(value) < 8:
        return "غير مضبوط"
    return f"{value[:4]}****{value[-4:]}"

for key_name in ["GROQ_API_KEY", "GOOGLE_API_KEY", "OPENAI_API_KEY"]:
    raw_val = os.environ.get(key_name)
    status = "موجود ومفعل" if raw_val else "غير مضاف حالياً"
    print(f"{key_name:15s} -> الحالة: {status:15s} | معاينة آمنة: {mask_key(raw_val)}")


GROQ_API_KEY    -> الحالة: موجود ومفعل     | معاينة آمنة: gsk_****44cJ
GOOGLE_API_KEY  -> الحالة: غير مضاف حالياً | معاينة آمنة: غير مضبوط
OPENAI_API_KEY  -> الحالة: غير مضاف حالياً | معاينة آمنة: غير مضبوط


### 3. جرب بنفسك!

دلوقتي دورك للتجربة الحرة:
1. جرب تغير اسم الملف أو تضيف متغير بيئي وهمي جديد عبر `os.environ["MY_TEST_KEY"] = "test_123456789"` وتشوف إزاي دالة `mask_key` هتغطيه!
2. تأكد دايماً إن ملف `.env` ومجلد `.venv/` متضافين جوا ملف `.gitignore` عندك عشان مفاتيحك ما تترفعش بالغلط على GitHub!


In [4]:
# جرب بنفسك هنا
os.environ["MY_PRACTICE_KEY"] = "sk_demo_987654321_secret"
print("معاينة المفتاح التجريبي بأمان:", mask_key(os.environ["MY_PRACTICE_KEY"]))

# فحص سريع لملف .gitignore لو موجود
gitignore_path = next((p / ".gitignore" for p in [Path("."), Path(".."), Path("../..")] if (p / ".gitignore").exists()), None)
if gitignore_path:
    content = gitignore_path.read_text(encoding="utf-8")
    print("هل .env محمي في .gitignore؟", ".env" in content)
    print("هل .venv محمي في .gitignore؟", ".venv" in content)


معاينة المفتاح التجريبي بأمان: sk_d****cret
هل .env محمي في .gitignore؟ True
هل .venv محمي في .gitignore؟ True
